# Part 1 — Python & Data Foundations: ingest, validate, clean, enrich

CareLoop Clinics — No-Show Guard

This notebook implements Tasks 1.1 through 1.6.

In [ ]:
# Imports
import csv
import json
import pandas as pd
import numpy as np
import requests
from pathlib import Path

DATA_DIR = Path("data")
APPOINTMENTS_CSV = DATA_DIR / "appointments.csv"
CLINICS_CSV = DATA_DIR / "clinics.csv"

print("Imports loaded. Data directory:", DATA_DIR.resolve())

## Task 1.1 — Read with `csv` module (5 marks)

Read appointments.csv using only csv.DictReader, parse numeric fields, separate accepted/rejected rows.

In [ ]:
def parse_row(row):
    """Convert age, fee_inr, prior_no_shows to int and distance_km to float."""
    converted = row.copy()
    converted["age"] = int(row["age"])
    converted["fee_inr"] = int(row["fee_inr"])
    converted["prior_no_shows"] = int(row["prior_no_shows"])
    converted["distance_km"] = float(row["distance_km"])
    return converted

accepted = []
rejected = []
total_rows = 0

with open(APPOINTMENTS_CSV, newline="", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    header = reader.fieldnames
    for row in reader:
        total_rows += 1
        try:
            parsed = parse_row(row)
            accepted.append(parsed)
        except ValueError as e:
            rejected.append({"row": row, "error": str(e)})

print(f"Header: {header}")
print(f"Total rows read: {total_rows}")
print(f"Accepted: {len(accepted)}")
print(f"Rejected: {len(rejected)}")

# Save rejected rows
with open("rejected_rows.json", "w", encoding="utf-8") as f:
    json.dump(rejected, f, indent=2)

print("Saved rejected_rows.json")

## Task 1.2 — Pandas cleaning pass 1 (5 marks)

Load with pandas, inspect, clean basics, drop bad rows. Start fresh from original CSV.

In [ ]:
df = pd.read_csv(APPOINTMENTS_CSV)

print(f"Raw shape: {df.shape}")
print(f"\nInfo:")
df.info()
print(f"\nNull counts:")
print(df.isnull().sum())
print(f"\nDuplicate rows: {df.duplicated().sum()}")
print(f"\nDistinct booking_channel raw: {df['booking_channel'].nunique()}")
print(df['booking_channel'].unique())

In [ ]:
# (a) Drop exact duplicate rows
df = df.drop_duplicates()
print(f"After drop_duplicates: {len(df)} rows")

# (b) Clean booking_channel
df["booking_channel"] = df["booking_channel"].str.strip().str.title()
print(f"Distinct booking_channel after cleaning: {df['booking_channel'].nunique()}")
print(df['booking_channel'].unique())

# (c) Convert dates: errors='coerce' ONLY for appt_date; booked_on without errors
df["appt_date"] = pd.to_datetime(df["appt_date"], errors="coerce")
df["booked_on"] = pd.to_datetime(df["booked_on"])

# (d) Drop rows where appt_date is missing or status is missing
before_drop = len(df)
df = df.dropna(subset=["appt_date", "status"])
print(f"After dropping missing appt_date/status: {len(df)} rows (dropped {before_drop - len(df)})")

## Task 1.3 — Cleaning pass 2: validity, gaps, outliers (5 marks)

Fix invalid values, fill missing, cap outliers using IQR. Order matters.

In [ ]:
# (a) Treat age < 1 or > 100 as invalid -> NaN
invalid_age_mask = (df["age"] < 1) | (df["age"] > 100)
invalid_count = invalid_age_mask.sum()
df.loc[invalid_age_mask, "age"] = np.nan
print(f"Invalid ages set to NaN: {invalid_count}")
print(f"Total missing ages now: {df['age'].isna().sum()}")

# (b) Fill missing age with median, distance_km with median
age_median = df["age"].median()
dist_median = df["distance_km"].median()
print(f"Median age: {age_median}")
print(f"Median distance_km: {dist_median}")

df["age"] = df["age"].fillna(age_median).round().astype(int)
df["distance_km"] = df["distance_km"].fillna(dist_median)

# (c) IQR capping for distance_km and fee_inr
for col in ["distance_km", "fee_inr"]:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    
    capped_low = (df[col] < lower).sum()
    capped_high = (df[col] > upper).sum()
    
    df[col] = np.clip(df[col], lower, upper)
    
    print(f"{col}: Q1={Q1:.4f}, Q3={Q3:.4f}, IQR={IQR:.4f}, lower={lower:.4f}, upper={upper:.4f}")
    print(f"  Capped low: {capped_low}, Capped high: {capped_high}")

# (d) Round distance_km to 1 decimal, fee_inr to int
df["distance_km"] = df["distance_km"].round(1)
df["fee_inr"] = df["fee_inr"].round().astype(int)
print(f"\nMax fee_inr after capping: {df['fee_inr'].max()}")

## Task 1.4 — Features and NumPy summary (4 marks)

Create lead_days and no_show, compute statistics with NumPy (no loops).

In [ ]:
# Create features
df["lead_days"] = (df["appt_date"] - df["booked_on"]).dt.days
df["no_show"] = (df["status"] == "No-Show").astype(int)

print(f"lead_days range: {df['lead_days'].min()} to {df['lead_days'].max()}")
print(f"No-shows: {df['no_show'].sum()} out of {len(df)} ({df['no_show'].mean()*100:.1f}%)")

# NumPy array
X = df[["age", "distance_km", "fee_inr", "lead_days"]].to_numpy()

# (a) Column means
col_means = X.mean(axis=0)
print(f"\nColumn means (age, distance_km, fee_inr, lead_days):")
print(col_means)

# (b) lead_days > 14
mask = X[:, 3] > 14
count_gt14 = mask.sum()
share_gt14 = mask.mean()
print(f"\nRows with lead_days > 14: {count_gt14} ({share_gt14*100:.1f}%)")

# (c) z-scores of distance_km
dist = X[:, 1]
z = (dist - np.mean(dist)) / np.std(dist)
count_z = (np.abs(z) > 2).sum()
print(f"Rows with |z| > 2 for distance_km: {count_z}")

## Task 1.5 — Enrich clinics with Open-Meteo API (3 marks)

Fetch temperature for each clinic's coordinates.

In [ ]:
clinics = pd.read_csv(CLINICS_CSV)
print(f"Clinics loaded: {len(clinics)} rows")
print(clinics)

temps = []
for _, row in clinics.iterrows():
    lat = row["latitude"]
    lon = row["longitude"]
    name = row["clinic_name"]
    
    temp_c = None
    for attempt in range(2):  # retry once
        try:
            resp = requests.get(
                "https://api.open-meteo.com/v1/forecast",
                params={
                    "latitude": lat,
                    "longitude": lon,
                    "hourly": "temperature_2m",
                    "forecast_days": 1
                },
                timeout=10
            )
            if resp.status_code == 200:
                data = resp.json()
                if "hourly" in data and "temperature_2m" in data["hourly"] and len(data["hourly"]["temperature_2m"]) > 0:
                    temp_c = data["hourly"]["temperature_2m"][0]
                    break
                else:
                    print(f"  {name}: missing hourly data in response")
            else:
                print(f"  {name}: HTTP {resp.status_code}")
        except requests.exceptions.RequestException as e:
            print(f"  {name}: request error ({e})")
    
    if temp_c is None:
        print(f"  {name}: FAILED after retries -> temp_c = None")
    else:
        print(f"  {name}: {temp_c}°C")
    
    temps.append(temp_c)

clinics["temp_c"] = temps
clinics.to_csv("clinics_enriched.csv", index=False)
print("\nSaved clinics_enriched.csv")
print(clinics)

## Task 1.6 — Export clean table and SQL load script (4 marks)

Sort, format dates, save appointments_clean.csv and load_data.sql

In [ ]:
# Verify before export
print(f"Unique appointment_id: {df['appointment_id'].nunique()}")
print(f"Total rows: {len(df)}")
print(f"Missing values per column:")
print(df.isnull().sum())

# Sort by appointment_id
df = df.sort_values("appointment_id").reset_index(drop=True)

# Format dates as YYYY-MM-DD strings
df["booked_on"] = df["booked_on"].dt.strftime("%Y-%m-%d")
df["appt_date"] = df["appt_date"].dt.strftime("%Y-%m-%d")

# Select exactly 16 columns in specified order
cols_order = [
    "appointment_id", "patient_id", "clinic_id", "booked_on", "appt_date",
    "age", "gender", "distance_km", "fee_inr", "booking_channel",
    "visit_type", "reminder_sent", "prior_no_shows", "status",
    "lead_days", "no_show"
]
df_clean = df[cols_order]

# Save appointments_clean.csv
df_clean.to_csv("appointments_clean.csv", index=False)
print(f"\nSaved appointments_clean.csv: {df_clean.shape}")
print(df_clean.head())

In [ ]:
# Write load_data.sql - each INSERT on ONE line
clinics_enriched = pd.read_csv("clinics_enriched.csv")

with open("load_data.sql", "w", encoding="utf-8") as f:
    # Clinics first (foreign key)
    for _, row in clinics_enriched.iterrows():
        temp_val = "NULL" if pd.isna(row["temp_c"]) else str(row["temp_c"])
        f.write(f"INSERT INTO clinics VALUES ('{row['clinic_id']}', '{row['clinic_name']}', '{row['city']}', '{row['tier']}', {row['latitude']}, {row['longitude']}, {temp_val});\n")
    
    # Appointments - each on one line
    for _, row in df_clean.iterrows():
        f.write(f"INSERT INTO appointments VALUES ('{row['appointment_id']}', '{row['patient_id']}', '{row['clinic_id']}', '{row['booked_on']}', '{row['appt_date']}', {row['age']}, '{row['gender']}', {row['distance_km']}, {row['fee_inr']}, '{row['booking_channel']}', '{row['visit_type']}', '{row['reminder_sent']}', {row['prior_no_shows']}, '{row['status']}', {row['lead_days']}, {row['no_show']});\n")

print("Saved load_data.sql")

# Count INSERT statements
with open("load_data.sql", "r") as f:
    lines = [l for l in f if l.strip().startswith("INSERT")]
print(f"Total INSERT statements: {len(lines)}")